# NIST Community Resilience — TraCR Exploratory Data Analysis

**Purpose:** Assess the structure, geographic and temporal coverage, and missing data in the NIST Tracking Community Resilience (TraCR) v1 database. The Orange County, Florida pilot supports the team's Orlando-focused community resilience project.

**Source:** [NIST TraCR dataset (version 1.0)](https://data.nist.gov/od/id/mds2-3978) · [Direct CSV download](https://data.nist.gov/od/ds/mds2-3978/TraCR_v1_database.csv) · [Metadata workbook](https://data.nist.gov/od/ds/mds2-3978/TraCR_Metadata.xlsx)

The notebook reads the **official CSV directly from NIST**, so no local data file is required. Running it requires an internet connection. Missing values indicate unavailable observations, not necessarily zeros. Coverage is a completeness measure, **not** a resilience score.


## 1. Setup and load the official TraCR CSV

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path

DATA_URL = "https://data.nist.gov/od/ds/mds2-3978/TraCR_v1_database.csv"
Path("figures").mkdir(exist_ok=True)

# Download directly from NIST; keep FIPS codes as strings (leading zeros matter).
df = pd.read_csv(DATA_URL, dtype={"fips": "string"}, low_memory=False)
print("TraCR shape:", df.shape)
display(df.head())
print("Column count:", len(df.columns))


## 2. Structure, temporal coverage, and indicator categories

In [ ]:
print("Unique geographic identifiers:", df["fips"].nunique())
print("Periods:", sorted(df["period"].dropna().unique()))
display(df.groupby("period")["fips"].nunique().rename("Geographic areas"))
print("Duplicate county-period rows:", df.duplicated(subset=["fips", "period"]).sum())


In [ ]:
df['period'].value_counts().sort_index().plot(kind='bar', figsize=(12,5))

plt.title("Number of Records by Year")
plt.xlabel("Year")
plt.ylabel("Number of Records")
plt.show()

In [ ]:
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

# Group TraCR measures by category
categories = {
    "Infrastructure": [c for c in df.columns if c.startswith("INFRA")],
    "Social Well-Being": [c for c in df.columns if c.startswith("SOCWB")],
    "Built Environment": [c for c in df.columns if c.startswith("BLTENV")],
    "Economic Vitality": [c for c in df.columns if c.startswith("ECNVIT")],
    "Natural Environment": [c for c in df.columns if c.startswith("NATENV")]
}

category_counts = pd.Series({
    category: len(columns)
    for category, columns in categories.items()
}).sort_values()

# MSU green
msu_green = "#18453B"

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.barh(
    category_counts.index,
    category_counts.values,
    color=msu_green,
    height=0.65
)

# Add values at the end of each bar
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.6,
        bar.get_y() + bar.get_height() / 2,
        f"{int(width)}",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

ax.set_title(
    "TraCR Resilience Measures by Category",
    fontsize=15,
    fontweight="bold",
    pad=16
)

ax.set_xlabel("Number of Measures", fontsize=11)
ax.set_xlim(0, category_counts.max() + 5)

# Clean presentation styling
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("figures/TraCR_Category_MSU.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Overall data completeness

In [ ]:
missing_summary = pd.DataFrame({
    'Missing': df.isna().sum(),
    'Available': df.notna().sum(),
    'Percent Missing': df.isna().mean() * 100,
    'Percent Available': df.notna().mean() * 100
})

missing_summary.sort_values('Percent Available', ascending=False)

In [ ]:
# Exclude identifiers and period from measures when assessing completeness.
measure_cols = [c for c in df.columns if c not in ["UID", "fips", "period"]]
coverage_all = df[measure_cols].notna().mean().mul(100)
print("Measures:", len(measure_cols))
print("Overall observed cells (%):", round(df[measure_cols].notna().to_numpy().mean() * 100, 2))
print("Measures entirely missing:", int((coverage_all == 0).sum()))
print("Measures with >=80% coverage:", int((coverage_all >= 80).sum()))
display(coverage_all.describe().rename("Coverage (%)").to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(coverage_all, bins=np.linspace(0, 100, 11), color="#18453B", edgecolor="white")
ax.set(title="Distribution of TraCR Measure Completeness", xlabel="Available observations per measure (%)", ylabel="Number of measures")
plt.tight_layout()
plt.show()


### Completeness by period

In [ ]:
period_coverage = df.groupby("period")[measure_cols].apply(lambda x: x.notna().to_numpy().mean() * 100)
display(period_coverage.rename("Available measure cells (%)").to_frame())
period_coverage.plot(marker="o", figsize=(10, 4), color="#18453B")
plt.title("TraCR Measure Completeness by Period")
plt.ylabel("Available measure cells (%)")
plt.xlabel("Period")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 4. Florida and Orange County pilot

Orange County (FIPS `12095`) is used as a county-level geographic pilot relevant to Orlando. TraCR observations are county-level; they should not be described as city-specific measurements.

In [ ]:
df["fips_clean"] = df["fips"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(5)
florida = df[df["fips_clean"].str.startswith("12", na=False)].copy()
orange = df[df["fips_clean"] == "12095"].copy()
print("Florida rows:", len(florida), "| Orange County rows:", len(orange))
print("Orange County periods:", sorted(orange["period"].dropna().unique()))
if orange.empty:
    raise ValueError("No Orange County observations found. Check FIPS formatting and dataset version.")


In [ ]:
measure_cols = [
    col for col in df.columns
    if col not in ["UID", "fips", "fips_clean", "period"]
]

orange_coverage = orange[measure_cols].notna().mean() * 100

print("Total TraCR measures:", len(measure_cols))
print("Measures with ANY Orange County data:",
      (orange_coverage > 0).sum())

print("Measures with >= 50% coverage:",
      (orange_coverage >= 50).sum())

print("Measures with 100% coverage:",
      (orange_coverage == 100).sum())

orange_coverage.sort_values(ascending=False).head(20)

In [ ]:
year_coverage = orange.set_index("period")[measure_cols].notna().sum(axis=1)

import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(
    year_coverage.index,
    year_coverage.values,
    marker="o"
)

plt.xlabel("Year")
plt.ylabel("Number of Measures Available")
plt.title("TraCR Data Availability in Orange County, Florida")
plt.xticks(year_coverage.index, rotation=45)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
Path("figures").mkdir(exist_ok=True)

categories = {
    "Infrastructure": "INFRA",
    "Social Well-Being": "SOCWB",
    "Built Environment": "BLTENV",
    "Economic Vitality": "ECNVIT",
    "Natural Environment": "NATENV"
}

coverage = {}

for category, prefix in categories.items():
    cols = [c for c in measure_cols if c.startswith(prefix)]
    coverage[category] = orange[cols].notna().mean().mean() * 100

coverage = pd.Series(coverage).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.5))

bars = ax.barh(
    coverage.index,
    coverage.values,
    color="#18453B",
    height=0.65
)

for bar in bars:
    ax.text(
        bar.get_width() + 0.6,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.1f}%",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

ax.set_title(
    "Average TraCR Data Coverage by Category (Orange County)",
    fontsize=15,
    fontweight="bold",
    pad=16
)
ax.set_xlabel("Average Coverage (%)")
ax.set_xlim(0, 55)
ax.grid(axis="x", alpha=0.15)
ax.set_axisbelow(True)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="y", length=0)

plt.tight_layout()
plt.savefig("figures/TraCR_Coverage_MSU.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Additional pilot comparison: Orange County vs. Florida

In [ ]:
# Compare completeness, not resilience outcomes.
florida_coverage = florida[measure_cols].notna().mean().mul(100)
comparison = pd.DataFrame({
    "Orange County (%)": orange[measure_cols].notna().mean().mul(100),
    "Florida (%)": florida_coverage
})
comparison["Difference (percentage points)"] = comparison["Orange County (%)"] - comparison["Florida (%)"]
display(comparison.sort_values("Difference (percentage points)").head(10))
print("Mean Orange County measure coverage (%):", round(comparison["Orange County (%)"].mean(), 1))
print("Mean Florida measure coverage (%):", round(comparison["Florida (%)"].mean(), 1))


### Coverage by resilience category: Orange County vs. Florida

In [ ]:
prefixes = {
    "Infrastructure": "INFRA",
    "Social Well-Being": "SOCWB",
    "Built Environment": "BLTENV",
    "Economic Vitality": "ECNVIT",
    "Natural Environment": "NATENV",
}
category_comparison = pd.DataFrame({
    "Orange County": {name: orange[[c for c in measure_cols if c.startswith(prefix)]].notna().to_numpy().mean() * 100 if any(c.startswith(prefix) for c in measure_cols) else np.nan for name, prefix in prefixes.items()},
    "Florida": {name: florida[[c for c in measure_cols if c.startswith(prefix)]].notna().to_numpy().mean() * 100 if any(c.startswith(prefix) for c in measure_cols) else np.nan for name, prefix in prefixes.items()}
})
display(category_comparison.round(1))
category_comparison.plot.bar(figsize=(9, 4), color=["#18453B", "#8BA89C"])
plt.title("Indicator Availability by Category")
plt.ylabel("Available observations (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 6. Findings, limitations, and next steps

- **What this analysis evaluates:** TraCR's coverage across counties, periods, indicator groups, and an Orange County pilot.
- **Data challenges:** Missing measures and coded indicators complicate direct comparison. Interpret each measure using NIST's metadata and source definitions before modeling or drawing substantive conclusions.
- **Geographic limitation:** Orange County is a proxy geographic area for an Orlando-focused pilot, **not** a dataset of City of Orlando observations.
- **Project fit:** TraCR provides quantitative county-level resilience context, but it does **not** directly identify or download municipal resilience planning documents.
- **Next steps:** Select well-documented indicators with adequate completeness, review their units and definitions in the [official metadata](https://data.nist.gov/od/ds/mds2-3978/TraCR_Metadata.xlsx), and evaluate whether they can supplement the team's document-discovery workflow.

**Interpretation note:** The charts describe availability, not actual resilience performance. Review the executed outputs before adding specific numeric findings to the team's report.
